# Pytree policies

`stringjax_tools` provides generic pytree mechanics, not package-specific
state policy.  Each consuming package should define its own static and ignored
attribute names, then reuse one policy across the classes that share those
conventions.


In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from stringjax_tools.pytrees import PytreePolicy, flatten_func, unflatten_func_class


## Shared policy registration

Static attributes travel as JAX auxiliary data.  Ignored attributes are dropped
from reconstructed traced objects by the base policy, so they must be limited to
recomputable caches or scratch state.  Semantic state, such as user-supplied
bounds or physical parameters, should be static if it is hashable and immutable,
or a traced child if it is array-like.


In [2]:
POLICY = PytreePolicy(static_keys=('scale',), ignore_keys=('_cache',))

@POLICY.register
class Model:
    def __init__(self, scale, values):
        self.scale = scale
        self.values = values
        self._cache = {'eager-only': True}

model = Model(scale=2.0, values=jnp.array([1.0, 2.0]))
updated = jax.tree_util.tree_map(lambda x: x + 1.0, model)

print('scale:', updated.scale)
print('values:', updated.values)
print('has cache:', hasattr(updated, '_cache'))


scale: 2.0
values: [2. 3.]
has cache: False


## Restoring ignored caches

JAX pytree reconstruction bypasses `__init__`.  If a class method may read an
ignored cache after a round trip, use a class-specific unflatten wrapper that
restores the cache to a safe default.  The example below is the intended pattern
behind a future `ignore_defaults={...}` policy option: true configuration is
static, while only the rebuildable cache is ignored and restored.


In [3]:
def _default_value(default):
    return default() if callable(default) else default


def make_restoring_flatteners(myclass, *, static_keys=(), ignore_defaults=None):
    ignore_defaults = dict(ignore_defaults or {})
    ignore_keys = tuple(ignore_defaults)

    def flatten(obj):
        return flatten_func(obj, static_keys=static_keys, ignore_keys=ignore_keys)

    def unflatten(aux_data, children):
        obj = unflatten_func_class(aux_data, children, myclass)
        for key, default in ignore_defaults.items():
            object.__setattr__(obj, key, _default_value(default))
        return obj

    return flatten, unflatten

In [4]:
class StatefulModel:
    def __init__(self, h12, user_Q=None, bounds=(-1, 1)):
        self.h12 = h12
        self._user_Q = user_Q
        self._sampler_config = (('bounds', tuple(bounds)),)
        self.values = jnp.ones(h12)
        self._sampler = None

    def Q(self):
        if self._user_Q is not None:
            return self._user_Q
        return self.h12 + 2

    @property
    def sampler(self):
        if self._sampler is None:
            self._sampler = dict(self._sampler_config)
        return self._sampler


flatten_stateful, unflatten_stateful = make_restoring_flatteners(
    StatefulModel,
    static_keys=('h12', '_user_Q', '_sampler_config'),
    ignore_defaults={'_sampler': None},
)
jax.tree_util.register_pytree_node(StatefulModel, flatten_stateful, unflatten_stateful)

stateful = StatefulModel(h12=2, user_Q=100, bounds=(-7, 7))
print('before Q:', stateful.Q())
print('before sampler:', stateful.sampler)

children, treedef = jax.tree_util.tree_flatten(stateful)
rebuilt_stateful = jax.tree_util.tree_unflatten(treedef, children)

print('after has _sampler:', hasattr(rebuilt_stateful, '_sampler'))
print('after _sampler:', rebuilt_stateful._sampler)
print('after Q:', rebuilt_stateful.Q())
print('after sampler:', rebuilt_stateful.sampler)

before Q: 100
before sampler: {'bounds': (-7, 7)}
after has _sampler: True
after _sampler: None
after Q: 100
after sampler: {'bounds': (-7, 7)}


## Why configuration should not be ignored

Restoring every ignored attribute to `None` is not a safe general fix.  If an
ignored field actually stores user configuration, a round trip can silently
change the model.  Keep configuration in static auxiliary data and reserve
ignored fields for caches that can be rebuilt from the preserved state.


In [5]:
class MisclassifiedModel:
    def __init__(self, h12, user_Q=None):
        self.h12 = h12
        self._user_Q = user_Q
        self.values = jnp.ones(h12)

    def Q(self):
        if self._user_Q is not None:
            return self._user_Q
        return self.h12 + 2


flatten_bad, unflatten_bad_base = make_restoring_flatteners(
    MisclassifiedModel,
    static_keys=('h12',),
    ignore_defaults={'_user_Q': None},
)
jax.tree_util.register_pytree_node(MisclassifiedModel, flatten_bad, unflatten_bad_base)

bad = MisclassifiedModel(h12=2, user_Q=100)
children, treedef = jax.tree_util.tree_flatten(bad)
rebuilt_bad = jax.tree_util.tree_unflatten(treedef, children)

print('before Q:', bad.Q())
print('after Q:', rebuilt_bad.Q())

before Q: 100
after Q: 4


## Direct flattening

For lower-level registration flows, the direct flatten/unflatten functions are
also available.  They take the package policy as explicit arguments.


In [6]:
children, aux = flatten_func(model, static_keys=('scale',), ignore_keys=('_cache',))
rebuilt = unflatten_func_class(aux, children, Model)

print('children:', len(children))
print('rebuilt scale:', rebuilt.scale)
print('rebuilt values:', rebuilt.values)


children: 1
rebuilt scale: 2.0
rebuilt values: [1. 2.]


Static auxiliary data is validated by default.  Non-hashable metadata should
not be static unless the consuming package has a clear reason and documents the
choice.


In [7]:
class BadStatic:
    def __init__(self):
        self.meta = {'not': 'hashable'}
        self.values = jnp.ones(2)

try:
    flatten_func(BadStatic(), static_keys=('meta',))
except ValueError as exc:
    print(type(exc).__name__ + ':', exc)


ValueError: Static pytree attribute 'meta' is not hashable. Use validate_static=False only if you know JAX can safely carry this value as auxiliary data, or remove it from static_keys.
